# Laboratory 3.6: Shor's Algorithm
## Factoring Large Numbers with Quantum Computers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Quantum-Plus-Labs/quantum-labs-2026/blob/main/lab3.6/lab3_6_shor.ipynb)

**Objective:** Implement the period-finding part of Shor's algorithm, the engine that can crack RSA encryption.

**Learning Objectives:**
1. Build and verify the Quantum Fourier Transform (QFT).
2. Implement modular exponentiation for a specific case ($N=15, a=7$).
3. Perform classical post-processing to extract factors from measurement results.

**The Problem:**
Given $N=15$, we want to find its factors (3 and 5). Shor's algorithm does this by finding the period $r$ of the function $f(x) = a^x \pmod{N}$.

Why it matters: the RSA encryption used on the web publishes a large number $N = p \times q$; whoever finds $p$ and $q$ can read the messages. For 15 we can do it by hand. The point of this notebook is to see *how* a quantum computer does it, because the same method scales to numbers far too large to factor classically.

---

### Environment Setup

In [ ]:
!pip install qiskit[visualization] qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc

import math
import fractions
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_histogram, plot_state_qsphere
from IPython.display import display

print("\n ENVIRONMENT SETUP COMPLETE!")

### Exercise 2a: Build and Verify the QFT

**What the QFT does, in one picture.** The Quantum Fourier Transform takes a basis state $|x\rangle$ of $n$ qubits and spreads it over *all* $2^n$ basis states with equal probability. The information about $x$ moves into the **phases** (the colours on the Q-sphere, as in Lab 1): the amplitude of $|k\rangle$ gets a phase of $x\cdot k/2^n$ of a full turn. For $x = 1$ on 3 qubits the phase grows by $1/8$ of a turn from one state to the next: a *phase ramp*. A larger $x$ makes the phase turn faster, so $x$ acts as a **frequency**. The inverse QFT goes the other way: it turns a regularly repeating phase pattern back into a sharp value, and Shor's algorithm uses exactly this.

**Your task:** run the cell. It first checks that the circuit is the Fourier transform (it prints `True`), then prints the phase of every amplitude of QFT$|001\rangle$. You should read 0/8, 1/8, …, 7/8 of a turn. **Bit order:** in Qiskit, qubit 0 is the *rightmost* bit (Lab 2), so `qc.x(0)` prepares $|001\rangle$ = 1.

In [ ]:
def build_qft(n):
    """Quantum Fourier Transform on n qubits.
    Qubit 0 is the least significant bit (Qiskit's order), so the integer x
    is read from a measured bitstring with int(bitstring, 2).
    Maps |x> to (1/sqrt(2^n)) * sum over k of exp(2*pi*i * x*k / 2^n) |k>."""
    qc = QuantumCircuit(n, name='QFT')
    for j in reversed(range(n)):            # start from the most significant qubit
        qc.h(j)
        for k in reversed(range(j)):        # controlled phase rotations from the lower qubits
            qc.cp(math.pi / 2 ** (j - k), k, j)
    for i in range(n // 2):                 # swaps put the output bits in the right order
        qc.swap(i, n - 1 - i)
    return qc

# 1) Verify: the circuit's matrix equals the Fourier-transform matrix
n_qubits = 3
M = 2 ** n_qubits
fourier = np.array([[np.exp(2j * np.pi * x * k / M) for x in range(M)]
                    for k in range(M)]) / np.sqrt(M)
print("QFT circuit equals the Fourier matrix:",
      np.allclose(Operator(build_qft(n_qubits)).data, fourier))

# 2) Apply the QFT to |001> (decimal 1) and look at the phases
qc_qft = QuantumCircuit(n_qubits)
qc_qft.x(0)                                  # qubit 0 = least significant bit -> |001>
qc_qft.compose(build_qft(n_qubits), inplace=True)
state = Statevector.from_instruction(qc_qft)

print("QFT|001>: every outcome has probability 1/8; the phase grows by 1/8 of a turn per step:")
for k, amp in enumerate(state.data):
    turns = (np.angle(amp) / (2 * np.pi)) % 1
    print(f"  |{k:03b}>  (k={k})  probability = {abs(amp)**2:.3f}   phase = {round(turns*8)}/8 of a turn")
display(plot_state_qsphere(state))
display(qc_qft.draw('mpl'))

### Exercise 2b: Shor's Period Finding ($N=15$, $a=7$)
**From factoring to a period.** Pick a number $a$ that shares no factor with $N$ (here $a = 7$). The powers $7^x \bmod 15$ (the remainder after dividing by 15) repeat in a cycle: $1, 7, 4, 13, 1, 7, \dots$ The length of the cycle is the **period** $r$; here $r = 4$. Since $7^4 - 1 = 2400 = 48 \times 50$ is divisible by 15 but neither 48 nor 50 is, the factors of 15 are shared between the two: $\gcd(48,15) = 3$ and $\gcd(50,15) = 5$ (gcd = greatest common divisor). In general $p, q = \gcd(a^{r/2} \pm 1, N)$, provided $r$ is even and $a^{r/2} \bmod N \neq N-1$.

**What the circuit does.** (1) The **counting register** (4 qubits) gets an H on each qubit, so it holds all $x = 0 \dots 15$ at once. (2) The **work register** (4 qubits) starts at the number 1. Counting qubit $j$ multiplies it by $7^{2^j} \bmod 15$ (by 7, then 4, then 1, then 1), so it ends up holding $7^x \bmod 15$, entangled with $x$ (Lab 2). (3) The **inverse QFT** acts on the counting register. The $x$ values that share the same $7^x \bmod 15$ form a comb spaced $r$ apart, and the inverse QFT turns that comb into peaks spaced $16/r$ apart. (4) We measure the counting register.

The powers $7^{2^j} \bmod 15$ are computed classically beforehand (by repeated squaring), as in any real run of Shor's algorithm. For a realistic $N$ it is extremely unlikely that any of them turns out to be 1; here the tiny $N = 15$ lets us see $7^4 \equiv 1$ directly — a reminder that this is a toy instance, not a way of knowing $r$ in advance.

**How to read the result.** Qiskit prints the counting register with qubit 0 on the right (Lab 2), so `'0100'` is the number 4 (`int('0100', 2)`). Expect four bars at 0, 4, 8, 12, each about 25 %.

**Your task:** run the cell. It first checks the multiplication gates on every input, then prints the exact probabilities and a seeded 1000-shot run.

In [ ]:
N, a = 15, 7
n_count = 4      # counting register: 4 qubits -> measured values y = 0 ... 15
n_work = 4       # work register: 4 qubits hold one number 0 ... 15

def c_mult_mod15(qc, ctrl, w, factor):
    """If qubit `ctrl` is 1, multiply the number held in the work qubits w by `factor`, modulo 15.
    w[0] is the least significant bit. Works for the values 1 ... 14, the only ones that occur here.
    Trick: doubling mod 15 moves every bit one place up, the top bit wrapping to the bottom, so
      x4 = move every bit two places (two swaps);
      x7 = x(-8) = move every bit three places, then flip all bits (15 - z = NOT z for 4 bits)."""
    if factor == 1:
        return                                  # multiplying by 1 changes nothing
    if factor == 4:
        qc.cswap(ctrl, w[0], w[2])
        qc.cswap(ctrl, w[1], w[3])
    elif factor == 7:
        qc.cswap(ctrl, w[0], w[1])
        qc.cswap(ctrl, w[1], w[2])
        qc.cswap(ctrl, w[2], w[3])
        for q in w:
            qc.cx(ctrl, q)
    else:
        raise ValueError("for a = 7 only the factors 1, 4 and 7 are needed")

# Check the arithmetic on every allowed input before trusting it
for factor in (7, 4):
    for y in range(1, 15):
        test = QuantumCircuit(5)                # qubit 0 = control, qubits 1-4 = work
        test.x(0)
        for b in range(4):
            if (y >> b) & 1:
                test.x(1 + b)
        c_mult_mod15(test, 0, [1, 2, 3, 4], factor)
        out = int(np.argmax(np.abs(Statevector(test).data))) >> 1
        assert out == (factor * y) % N
print("Checked: the gates multiply by 7 and by 4 modulo 15 on every input 1 ... 14.")

def build_shor_circuit():
    c = QuantumRegister(n_count, 'count')
    w = QuantumRegister(n_work, 'work')
    cr = ClassicalRegister(n_count, 'c')
    qc = QuantumCircuit(c, w, cr)

    qc.h(c)            # counting register: equal superposition of x = 0 ... 15
    qc.x(w[0])         # work register starts at the number 1 (|0001>)
    # Counting qubit j multiplies the work register by a^(2^j) mod 15.
    # Together they multiply by a^x, so the work register ends up holding 7^x mod 15.
    #   j=0: 7^1 = 7    j=1: 7^2 = 49 = 4 (mod 15)    j=2: 7^4 = 1    j=3: 7^8 = 1
    for j in range(n_count):
        c_mult_mod15(qc, c[j], w, pow(a, 2 ** j, N))
    qc.barrier()
    qc.compose(build_qft(n_count).inverse(), qubits=c, inplace=True)   # inverse QFT
    qc.measure(c, cr)
    return qc

qc_shor = build_shor_circuit()
exact = Statevector(qc_shor.remove_final_measurements(inplace=False)).probabilities_dict(qargs=range(n_count))
print("Exact probabilities:", {str(k): round(float(v), 3) for k, v in sorted(exact.items()) if v > 1e-9})

counts = AerSimulator(seed_simulator=1234).run(qc_shor, shots=1000).result().get_counts()
print("Shor results (1000 shots, seeded):", dict(sorted(counts.items())))
display(qc_shor.draw('mpl', fold=-1))
display(plot_histogram(counts))

### Exercise 2c: From a measurement to the factors
**Why this step:** each measured value $y$ satisfies $y/16 = s/r$ for some whole number $s$. Writing $y/16$ in lowest terms gives a denominator equal to $r$, or to a divisor of $r$. `Fraction(...).limit_denominator` does this; for large $N$ the same job is done by the continued-fractions algorithm. We then check $a^r \bmod N = 1$ and, if the check fails, try multiples of $r$. $y = 0$ carries no information, so we run again.

**Expected:** 0 → run again; 4, 8, 12 → $r = 4$ → factors 3 and 5, so about 75 % of runs succeed.

In [ ]:
print(f"{'bits':>6} {'y':>3} {'y/16':>6} {'fraction':>9}   result")
factors = set()
for bits in sorted(counts, key=lambda b: int(b, 2)):
    y = int(bits, 2)                                     # counting register as an integer
    frac = fractions.Fraction(y, 2 ** n_count).limit_denominator(N - 1)
    r = frac.denominator                                 # candidate period
    if y == 0:
        result = "phase 0 carries no information -> run again"
    else:
        note = ""
        while pow(a, r, N) != 1 and r < N:               # denominator too small? try its multiples
            note = f"{a}^{frac.denominator} mod {N} = {pow(a, frac.denominator, N)}, not 1 -> try multiples; "
            r += frac.denominator
        if pow(a, r, N) != 1:
            result = "no period found from this outcome -> run again"
        elif r % 2 == 1 or pow(a, r // 2, N) == N - 1:
            result = f"r = {r} cannot be used -> choose another a"
        else:
            p, q = math.gcd(a ** (r // 2) - 1, N), math.gcd(a ** (r // 2) + 1, N)
            result = note + f"r = {r}:  gcd({a}^{r//2} - 1, {N}) = {p},  gcd({a}^{r//2} + 1, {N}) = {q}"
            factors |= {p, q}
    print(f"{bits:>6} {y:>3} {str(y)+'/16':>6} {str(frac):>9}   {result}   [{counts[bits]} shots]")

print("\nFactors found:", sorted(factors), "->", " x ".join(map(str, sorted(factors))), "=", math.prod(factors))